In [15]:
import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), "D:\Project\IndianSignLanguage"))
sys.path.insert(0, project_root)

print("Project root:", project_root)  

Project root: D:\Project\IndianSignLanguage


In [16]:
import pandas as pd
import numpy as np
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
from src.utils import save_object, load_object
from src.logger import logging
from src.exception import CustomException

In [17]:
DATA_CSV   = "data/isl_keypoints.csv"
MODEL_DIR  = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "isl_model.pkl")
ENC_PATH   = os.path.join(MODEL_DIR, "label_encoder.pkl")
SCALER_PATH = os.path.join(MODEL_DIR, "scaler.pkl")

In [18]:
def train():
    try:
        logging.info("Loading dataset...")
        df = pd.read_csv("D:\Project\IndianSignLanguage\data\isl_keypoints.csv")
        print(f"Dataset: {df.shape[0]} rows, {df['label'].nunique()} classes")
        print(f"Classes: {sorted(df['label'].unique())}")

        X = df.drop("label", axis=1).values.astype(np.float32)
        y = df["label"].values

        #Encoding A-Z to 0-25
        le = LabelEncoder()
        y_enc = le.fit_transform(y)

        #Scaling features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
        )
        print(f"Train: {len(X_train)} | Test: {len(X_test)}")

        #MLP Neural Network
        print("\nTraining ANN... (this takes 1-3 mins)")
        model = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=42,
            verbose=True
        )

        model.fit(X_train, y_train)

        #Evaluate
        y_pred = model.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)

        print(f"\n Test Accuracy: {acc*100:.2f}%")
        print(classification_report(y_test, y_pred,
                                    target_names=le.classes_))
        logging.info(f"Test accuracy: {acc*100:.2f}%")

        #Save everything
        os.makedirs(MODEL_DIR, exist_ok=True)
        save_object(MODEL_PATH,  model)
        save_object(ENC_PATH,    le)
        save_object(SCALER_PATH, scaler)

        print(f"\nModel saved   → {MODEL_PATH}")
        print(f"Encoder saved → {ENC_PATH}")
        print(f"Scaler saved  → {SCALER_PATH}")

    except Exception as e:
        raise CustomException(e, sys)

if __name__ == "__main__":
    train()

Dataset: 5277 rows, 26 classes
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Train: 4221 | Test: 1056

Training ANN... (this takes 1-3 mins)
Iteration 1, loss = 2.62055167
Validation score: 0.434988
Iteration 2, loss = 1.42516125
Validation score: 0.676123
Iteration 3, loss = 0.84612811
Validation score: 0.784870
Iteration 4, loss = 0.61322076
Validation score: 0.787234
Iteration 5, loss = 0.50328390
Validation score: 0.817967
Iteration 6, loss = 0.43050237
Validation score: 0.839243
Iteration 7, loss = 0.37663005
Validation score: 0.862884
Iteration 8, loss = 0.32789697
Validation score: 0.865248
Iteration 9, loss = 0.30009709
Validation score: 0.855792
Iteration 10, loss = 0.27053178
Validation score: 0.895981
Iteration 11, loss = 0.24467437
Validation score: 0.881797
Iteration 12, loss = 0.22609137
Validation score: 0.900709
Iteration 13, loss = 0.20567482
Validation score: 0.895981
Iterati